# Group E — Ground Truth Comparison

Compares the learned boundaries against the **true action schemas** from the PDDL
domain file. This is the end-to-end correctness check.

### Expected relationships (before convergence)

| Boundary | Relationship to truth | Direction | Reason |
|---|---|---|---|
| `L_pre` | ⊇ true preconditions | Conservative | Sound model requires too much (spurious literals are expected) |
| `U_pre` (each hyp) | ⊆ true preconditions | Permissive | Complete model requires too little (missing literals are expected) |
| `L_eff` | ⊆ true effects | Conservative | Sound model predicts too little (missing effects are expected) |
| `U_eff` | ⊇ true effects | Permissive | Complete model predicts too much (spurious effects are expected) |

At **convergence** (`L_pre == U_pre` and `L_eff == U_eff`), all four boundaries
should equal the true model exactly.

### Key challenge: lifting

Ground truth from PDDL uses parameter names (e.g. `x`, `y` — UP drops the `?`).
The learner also stores lifted literals with the same parameter names, so comparison
is direct. Inequality constraints (`not (= ?x ?y)`) in the extended domain are
ignored — they are not fluent literals and are not stored by the learner.

Reference: `AlgorithmExplained.md §13 Group E`

In [1]:
import sys, os, tempfile, shutil, random as _random

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

from unified_planning.shortcuts import *
from unified_planning.io import PDDLReader
import unified_planning
get_environment().credits_stream = None

from ascal.learner import Learner
from ascal.models import Literal, State, Action, Demonstration
from ascal.transitions import generate_lifted_demonstrations_from_problem

## Setup — Mockup Domain

In [2]:
MOCKUP_DIR   = os.path.join('..', 'benchmarks', 'mockup')
DOMAIN_FILE  = os.path.join(MOCKUP_DIR, 'domain.pddl')
PROBLEM_FILE = os.path.join(MOCKUP_DIR, 'problems', 'problem-00.pddl')

assert os.path.exists(DOMAIN_FILE),  f'Domain not found: {DOMAIN_FILE}'
assert os.path.exists(PROBLEM_FILE), f'Problem not found: {PROBLEM_FILE}'
print(f'Domain:  {DOMAIN_FILE}')
print(f'Problem: {PROBLEM_FILE}')

Domain:  ../benchmarks/mockup/domain.pddl
Problem: ../benchmarks/mockup/problems/problem-00.pddl


In [3]:
reader  = PDDLReader()
problem = reader.parse_problem(DOMAIN_FILE, PROBLEM_FILE)

all_fluents    = problem.fluents
all_actions    = problem.actions
static_fluents = problem.get_static_fluents()
action_names   = [a.name for a in all_actions]

print('Actions :', action_names)
print('Fluents :', [f.name for f in all_fluents])
print('Static  :', [f.name for f in static_fluents])

Actions : ['pickup']
Fluents : ['on', 'on_table', 'clear', 'arm_empty', 'holding']
Static  : ['on']


## Helpers

### Ground truth extraction

`extract_ground_truth` converts the UP action schema into the same
`frozenset[Literal]` representation used by the learner.

UP parses `(and (clear ?x) (on_table ?x) ...)` as a **single AND node** inside
`action.preconditions`, not as a list of individual atoms. We therefore flatten
AND nodes recursively before extracting literals.

Only **fluent** preconditions are extracted — equality constraints
(`not (= ?x ?y)`) are skipped because they are not stored as literals.

In [4]:
def extract_ground_truth(action):
    """Return (true_pre, true_eff) as frozenset[Literal] from a UP action.

    Parameter names come from action.parameters (e.g. 'x', 'y', 'z').
    Inequality constraints (not (= ?x ?y)) are silently skipped.

    UP may return a single AND node in action.preconditions rather than
    individual atoms, so we flatten AND nodes recursively first.
    """
    def _args(node):
        result = []
        for a in node.args:
            if a.is_parameter_exp():
                result.append(a.parameter().name)
            elif a.is_object_exp():
                result.append(a.object().name)
        return tuple(result)

    def _flatten(node):
        """Yield individual atom-level FNodes, unwrapping AND nodes."""
        if node.is_and():
            for arg in node.args:
                yield from _flatten(arg)
        else:
            yield node

    true_pre = set()
    for cond in action.preconditions:
        for atom in _flatten(cond):
            if atom.is_fluent_exp():
                true_pre.add(Literal(atom.fluent().name, _args(atom), True))
            elif atom.is_not() and atom.arg(0).is_fluent_exp():
                inner = atom.arg(0)
                true_pre.add(Literal(inner.fluent().name, _args(inner), False))
            # else: equality constraint or other compound formula — skip

    true_eff = set()
    for eff in action.effects:
        fluent_node = eff.fluent
        fluent_name = fluent_node.fluent().name
        val         = bool(eff.value.constant_value())
        true_eff.add(Literal(fluent_name, _args(fluent_node), val))

    return frozenset(true_pre), frozenset(true_eff)


def fmt_lits(fset, indent='  '):
    if not fset:
        return indent + '\u2205  (empty)'
    return '\n'.join(indent + str(l) for l in sorted(str(l) for l in fset))


print('extract_ground_truth() and fmt_lits() defined.')

extract_ground_truth() and fmt_lits() defined.


### Sanity-check: inspect raw ground truth before running comparisons

In [5]:
print('=== Ground truth sanity check (mockup domain) ===')
for a in all_actions:
    tp, te = extract_ground_truth(a)
    print(f'\nAction: {a.name}({", ".join(p.name for p in a.parameters)})')
    print(f'  Preconditions ({len(tp)}):')
    print(fmt_lits(tp, '    '))
    print(f'  Effects ({len(te)}):')
    print(fmt_lits(te, '    '))

=== Ground truth sanity check (mockup domain) ===

Action: pickup(x)
  Preconditions (3):
    arm_empty()
    clear(x)
    on_table(x)
  Effects (4):
    holding(x)
    ¬arm_empty()
    ¬clear(x)
    ¬on_table(x)


In [6]:
def generate_demos(prob, max_neg_per_step=50, seed=0):
    """Delegates to :func:`ascal.transitions.generate_lifted_demonstrations_from_problem`."""
    return generate_lifted_demonstrations_from_problem(
        prob,
        max_neg_per_step=max_neg_per_step,
        max_check_per_action=None,
        seed=seed,
        verbose=False,
    )


print('generate_demos() defined.')

generate_demos() defined.


## Comparison Function

For each action, compares the four learned boundaries to the ground truth and
prints a detailed report:

- **Spurious** literals are in the learned boundary but not in the truth.
- **Missing** literals are in the truth but not in the learned boundary.

Which of these is a *bug* depends on which boundary:

| Boundary | Spurious | Missing |
|---|---|---|
| `L_pre` (sound, conservative) | Expected — model over-requires | **Bug** — model under-requires |
| `U_pre` (complete, permissive) | **Bug** — model over-requires | Expected — model under-requires |
| `L_eff` (sound, conservative) | **Bug** — model over-predicts | Expected — model under-predicts |
| `U_eff` (complete, permissive) | Expected — model over-predicts | **Bug** — model under-predicts |

Hard assertions are run only on the **bug** directions.

In [7]:
def _stat(count, is_violation_direction):
    """Format a count with a clear visual indicator.

    is_violation_direction=True  -> non-zero count is a real problem (❌)
    is_violation_direction=False -> non-zero count is expected (→)
    Zero is always ✓.
    """
    if count == 0:
        return f'{count} \u2713'
    return f'{count} \u274c' if is_violation_direction else f'{count} \u2192'


def run_group_e(fluents, actions, static, pos_demos, neg_demos, label=''):
    """Train on all demos then compare each boundary to ground truth.

    Returns the number of violations found.
    Violations are literals in the wrong direction (e.g. spurious L_eff,
    missing U_eff). Expected deviations are shown but never counted.
    """
    learner = Learner(fluents, actions, static)

    # Build interleaved sequence
    if not pos_demos:
        all_demos = list(neg_demos)
    elif not neg_demos:
        all_demos = list(pos_demos)
    else:
        slice_size = len(neg_demos) / len(pos_demos)
        all_demos  = []
        for i, pos in enumerate(pos_demos):
            all_demos.extend(neg_demos[int(slice_size * i):int(slice_size * (i + 1))])
            all_demos.append(pos)

    for demo in all_demos:
        learner.update(demo)

    if label:
        print(f'\n{"=" * 70}')
        print(f'  {label}')
        print(f'  Trained on: {len(pos_demos)} pos + {len(neg_demos)} neg demos')
        print(f'  Converged : {learner.converged}')
        print(f'{"=" * 70}')

    total_violations = 0

    for action in actions:
        n = action.name
        true_pre, true_eff = extract_ground_truth(action)

        hL_pre     = next(iter(learner.L_pre[n]))
        hL_eff     = next(iter(learner.L_eff[n]))
        hU_eff     = next(iter(learner.U_eff[n]))
        u_pre_hyps = list(learner.U_pre[n])

        vs        = learner.version_space_size[n]
        converged = vs['converged']
        eff_exp   = vs['n_eff'].bit_length() - 1 if vs['n_eff'] > 0 else 0
        conv_sym  = ('\u2705 converged' if converged else
                     f'\u23f3 not yet (|U_pre|={vs["n_pre"]}, eff uncertainty=2^{eff_exp})')

        print(f'\n\u2500\u2500\u2500 {n}  [{conv_sym}]  \u2500\u2500\u2500')

        # ─── Preconditions ──────────────────────────────────────────────────
        print('\nPRECONDITIONS')
        print(f'  True ({len(true_pre)} lits):')
        print(fmt_lits(true_pre, '    '))

        # L_pre: should be superset of true_pre
        #   spurious = expected (sound model over-requires)
        #   missing  = violation (sound model dropped a real precondition)
        l_pre_spurious = hL_pre - true_pre
        l_pre_missing  = true_pre - hL_pre
        print(f'\n  L_pre / sound ({len(hL_pre)} lits)  '
              f'[spurious={_stat(len(l_pre_spurious), False)}  '
              f'missing={_stat(len(l_pre_missing), True)}]')
        if l_pre_spurious:
            print('    \u2192 Spurious (expected — sound model is conservative):')
            print(fmt_lits(l_pre_spurious, '      '))
        if l_pre_missing:
            print('    \u274c Missing (violation — sound model dropped a required precondition):')
            print(fmt_lits(l_pre_missing, '      '))
            total_violations += len(l_pre_missing)
        if not l_pre_missing:
            print('    \u2713 No missing preconditions')

        # U_pre: each hyp should be subset of true_pre
        #   spurious = violation (complete model over-requires)
        #   missing  = expected (complete model under-requires / permissive)
        print(f'\n  U_pre / complete ({len(u_pre_hyps)} hyps)')
        for hi, hp in enumerate(u_pre_hyps):
            u_spurious = hp - true_pre
            u_missing  = true_pre - hp
            tag = f'hyp {hi}' if len(u_pre_hyps) > 1 else 'single hyp'
            print(f'    [{tag}] ({len(hp)} lits)  '
                  f'[spurious={_stat(len(u_spurious), True)}  '
                  f'missing={_stat(len(u_missing), False)}]')
            if u_spurious:
                print('      \u274c Spurious (violation — complete model over-requires):')
                print(fmt_lits(u_spurious, '        '))
                total_violations += len(u_spurious)
            if u_missing:
                print('      \u2192 Missing (expected — complete model is permissive):')
                print(fmt_lits(u_missing, '        '))
            if not u_spurious:
                print('      \u2713 No spurious preconditions')

        # ─── Effects ────────────────────────────────────────────────────────
        print('\nEFFECTS')
        print(f'  True ({len(true_eff)} lits):')
        print(fmt_lits(true_eff, '    '))

        # L_eff: should be subset of true_eff
        #   spurious = violation (sound model invented an effect)
        #   missing  = expected (sound model is conservative)
        l_eff_spurious = hL_eff - true_eff
        l_eff_missing  = true_eff - hL_eff
        print(f'\n  L_eff / sound ({len(hL_eff)} lits)  '
              f'[spurious={_stat(len(l_eff_spurious), True)}  '
              f'missing={_stat(len(l_eff_missing), False)}]')
        if l_eff_spurious:
            print('    \u274c Spurious (violation — sound model invented an effect):')
            print(fmt_lits(l_eff_spurious, '      '))
            total_violations += len(l_eff_spurious)
        if l_eff_missing:
            print('    \u2192 Missing (expected — sound model is conservative):')
            print(fmt_lits(l_eff_missing, '      '))
        if not l_eff_spurious:
            print('    \u2713 No spurious effects')

        # U_eff: should be superset of true_eff
        #   spurious = expected (complete model over-predicts / permissive)
        #   missing  = violation (complete model dropped a true effect)
        u_eff_spurious = hU_eff - true_eff
        u_eff_missing  = true_eff - hU_eff
        print(f'\n  U_eff / complete ({len(hU_eff)} lits)  '
              f'[spurious={_stat(len(u_eff_spurious), False)}  '
              f'missing={_stat(len(u_eff_missing), True)}]')
        if u_eff_missing:
            print('    \u274c Missing (violation — complete model dropped a true effect):')
            print(fmt_lits(u_eff_missing, '      '))
            total_violations += len(u_eff_missing)
        if u_eff_spurious:
            print('    \u2192 Spurious (expected — complete model is permissive):')
            print(fmt_lits(u_eff_spurious, '      '))
        if not u_eff_missing:
            print('    \u2713 No missing effects')

        # ─── Convergence check ─────────────────────────────────────────────
        if converged:
            print('\n  \u2714 CONVERGED — asserting exact match with ground truth')
            pre_match = hL_pre == true_pre
            eff_match = hL_eff == true_eff
            if pre_match and eff_match:
                print('    \u2705 Preconditions: exact match')
                print('    \u2705 Effects      : exact match')
            else:
                if not pre_match:
                    print('    \u274c Preconditions do NOT match ground truth after convergence!')
                    total_violations += 1
                if not eff_match:
                    print('    \u274c Effects do NOT match ground truth after convergence!')
                    total_violations += 1
        else:
            pre_gap = len(hL_pre) - len(true_pre)
            eff_gap = len(hU_eff) - len(hL_eff)
            print(f'\n  \u23f3 Remaining gap to ground truth:')
            print(f'    L_pre has {pre_gap} extra literal(s)  (spurious — will shrink with more demos)')
            print(f'    U_eff has {eff_gap} uncertain effect(s) (will shrink with more demos)')

    print(f'\n{"=" * 70}')
    if total_violations == 0:
        print('\u2705 All guarantee directions satisfied — no violations found.')
    else:
        print(f'\u274c {total_violations} violation(s) found.')

    return total_violations


print('run_group_e() defined.')


run_group_e() defined.


## Mockup — Run

In [8]:
tmpdir   = tempfile.mkdtemp(prefix='ascal_testE_')
orig_cwd = os.getcwd()
os.chdir(tmpdir)
try:
    pos_demos, neg_demos = generate_demos(problem)
finally:
    os.chdir(orig_cwd)
    shutil.rmtree(tmpdir, ignore_errors=True)

print(f'Total: {len(pos_demos)} positive, {len(neg_demos)} negative')

Total: 1 positive, 0 negative


In [9]:
run_group_e(
    all_fluents, all_actions, static_fluents,
    pos_demos, neg_demos,
    label='Mockup domain'
)


  Mockup domain
  Trained on: 1 pos + 0 neg demos
  Converged : False

─── pickup  [⏳ not yet (|U_pre|=16, eff uncertainty=2^0)]  ───

PRECONDITIONS
  True (3 lits):
    arm_empty()
    clear(x)
    on_table(x)

  L_pre / sound (4 lits)  [spurious=1 →  missing=0 ✓]
    → Spurious (expected — sound model is conservative):
      ¬holding(x)
    ✓ No missing preconditions

  U_pre / complete (1 hyps)
    [single hyp] (0 lits)  [spurious=0 ✓  missing=3 →]
      → Missing (expected — complete model is permissive):
        arm_empty()
        clear(x)
        on_table(x)
      ✓ No spurious preconditions

EFFECTS
  True (4 lits):
    holding(x)
    ¬arm_empty()
    ¬clear(x)
    ¬on_table(x)

  L_eff / sound (4 lits)  [spurious=0 ✓  missing=0 ✓]
    ✓ No spurious effects

  U_eff / complete (4 lits)  [spurious=0 ✓  missing=0 ✓]
    ✓ No missing effects

  ⏳ Remaining gap to ground truth:
    L_pre has 1 extra literal(s)  (spurious — will shrink with more demos)
    U_eff has 0 uncertain eff

0

## Scale-up: Blocks Domain (4 actions, 20 problems)

With 20 problems the version space likely has **not converged** — the algorithm
has not seen enough *diverse* demonstrations to fully pin down every boundary.
The report below shows the gap between the current state and the true model,
making clear what remains to be learned.

In [10]:
BLOCKS_DIR    = os.path.join('..', 'benchmarks', 'blocks')
BLOCKS_DOMAIN = os.path.join(BLOCKS_DIR, 'domain_extended.pddl')
BLOCKS_PROBS  = os.path.join(BLOCKS_DIR, 'problems')

assert os.path.exists(BLOCKS_DOMAIN), f'Blocks domain not found: {BLOCKS_DOMAIN}'

problem_files = sorted(f for f in os.listdir(BLOCKS_PROBS) if f.endswith('.pddl'))
print(f'Blocks domain : {BLOCKS_DOMAIN}')
print(f'Problem files : {len(problem_files)}')

blocks_reader  = PDDLReader()
blocks_problem = blocks_reader.parse_problem(
    BLOCKS_DOMAIN, os.path.join(BLOCKS_PROBS, problem_files[0]))

b_fluents      = blocks_problem.fluents
b_actions      = blocks_problem.actions
b_static       = blocks_problem.get_static_fluents()
b_action_names = [a.name for a in b_actions]
print(f'Actions       : {b_action_names}')

Blocks domain : ../benchmarks/blocks/domain_extended.pddl
Problem files : 21
Actions       : ['pickup', 'putdown', 'stack', 'unstack']


In [11]:
# Collect unique lifted demos (incremental dedup + max_neg_per_step cap)
seen_pos, seen_neg     = set(), set()
all_pos_blocks, all_neg_blocks = [], []

for pf in problem_files:
    prob = blocks_reader.parse_problem(BLOCKS_DOMAIN, os.path.join(BLOCKS_PROBS, pf))
    tmpdir   = tempfile.mkdtemp(prefix='ascal_testE_blocks_')
    orig_cwd = os.getcwd()
    os.chdir(tmpdir)
    try:
        p, n = generate_demos(prob)
    finally:
        os.chdir(orig_cwd)
        shutil.rmtree(tmpdir, ignore_errors=True)

    new_p = new_n = 0
    for d in p:
        k = repr(d)
        if k not in seen_pos:
            seen_pos.add(k); all_pos_blocks.append(d); new_p += 1
    for d in n:
        k = repr(d)
        if k not in seen_neg:
            seen_neg.add(k); all_neg_blocks.append(d); new_n += 1

    print(f'  {pf}: +{new_p} new pos  +{new_n} new neg  '
          f'(unique so far: {len(all_pos_blocks)} pos, {len(all_neg_blocks)} neg)')

print(f'\nFinal unique: {len(all_pos_blocks)} pos, {len(all_neg_blocks)} neg')

  problem-00.pddl: +13 new pos  +548 new neg  (unique so far: 13 pos, 548 neg)
  problem-01.pddl: +9 new pos  +318 new neg  (unique so far: 22 pos, 866 neg)
  problem-02.pddl: +10 new pos  +266 new neg  (unique so far: 32 pos, 1132 neg)
  problem-03.pddl: +12 new pos  +113 new neg  (unique so far: 44 pos, 1245 neg)
  problem-04.pddl: +13 new pos  +261 new neg  (unique so far: 57 pos, 1506 neg)
  problem-05.pddl: +4 new pos  +47 new neg  (unique so far: 61 pos, 1553 neg)
  problem-06.pddl: +1 new pos  +39 new neg  (unique so far: 62 pos, 1592 neg)
  problem-07.pddl: +0 new pos  +20 new neg  (unique so far: 62 pos, 1612 neg)
  problem-08.pddl: +1 new pos  +30 new neg  (unique so far: 63 pos, 1642 neg)
  problem-09.pddl: +2 new pos  +23 new neg  (unique so far: 65 pos, 1665 neg)
  problem-10.pddl: +0 new pos  +3 new neg  (unique so far: 65 pos, 1668 neg)
  problem-11.pddl: +0 new pos  +0 new neg  (unique so far: 65 pos, 1668 neg)
  problem-12.pddl: +0 new pos  +0 new neg  (unique so far: 

In [12]:
run_group_e(
    b_fluents, b_actions, b_static,
    all_pos_blocks, all_neg_blocks,
    label='Blocks domain (extended, 20 problems)'
)


  Blocks domain (extended, 20 problems)
  Trained on: 67 pos + 1677 neg demos
  Converged : False

─── pickup  [⏳ not yet (|U_pre|=128, eff uncertainty=2^6)]  ───

PRECONDITIONS
  True (3 lits):
    arm_empty()
    clear(x)
    on_table(x)

  L_pre / sound (10 lits)  [spurious=7 →  missing=0 ✓]
    → Spurious (expected — sound model is conservative):
      ¬holding(x)
      ¬holding(y)
      ¬holding(z)
      ¬on(x, y)
      ¬on(x, z)
      ¬on(y, x)
      ¬on(z, x)
    ✓ No missing preconditions

  U_pre / complete (1 hyps)
    [single hyp] (3 lits)  [spurious=0 ✓  missing=0 ✓]
      ✓ No spurious preconditions

EFFECTS
  True (4 lits):
    holding(x)
    ¬arm_empty()
    ¬clear(x)
    ¬on_table(x)

  L_eff / sound (4 lits)  [spurious=0 ✓  missing=0 ✓]
    ✓ No spurious effects

  U_eff / complete (10 lits)  [spurious=6 →  missing=0 ✓]
    → Spurious (expected — complete model is permissive):
      ¬holding(y)
      ¬holding(z)
      ¬on(x, y)
      ¬on(x, z)
      ¬on(y, x)
      ¬o

0

---

## Evaluation Method Comparison

Now that we know the ground truth (from the boundary comparison above), we can
run both evaluation methods and judge which is more accurate.

`run_group_e` tells us the **direction** the learned model is in relative to the
truth. The two evaluation methods tell us the **numeric P/R/F1** as seen by a
classifier that doesn't have access to the truth.

| Metric | evaluate() | evaluate_repr() |
|---|---|---|
| Complete recall C2 | Existential — passes if **any** hyp in U_pre covers the demo | Binary — only the most-general `repr_hp` must cover the demo |
| Complete precision | Fractional — averages over all hyps for negatives | Binary — repr_hp alone decides accept/reject |

If `evaluate_repr()` C2 < 1.0 before convergence, it means the representative
hypothesis is not always the right one — the algorithm's guarantee only covers
**existence** of a consistent hypothesis, not the specific shortest one.


In [13]:
from ascal.learner import Learner

def _compare_evaluations(fluents, actions, static, pos_demos, neg_demos, label=''):
    """Train on all demos then compare evaluate() vs evaluate_repr() side by side."""
    learner = Learner(fluents, actions, static)

    # Interleaved training
    if not pos_demos:
        seq = list(neg_demos)
    elif not neg_demos:
        seq = list(pos_demos)
    else:
        sz = len(neg_demos) / len(pos_demos)
        seq = []
        for i, pos in enumerate(pos_demos):
            seq.extend(neg_demos[int(sz * i):int(sz * (i + 1))])
            seq.append(pos)

    for demo in seq:
        learner.update(demo)

    if label:
        print(f'\n{"=" * 65}')
        print(f'  {label}')
        print(f'{"=" * 65}')

    # Run both methods
    f1_s,  f1_c,  p_s,  r_s,  p_c,  r_c         = learner.evaluate(pos_demos, neg_demos)
    f1_s2, f1_c2, p_s2, r_s2, p_c2, r_c2, status = learner.evaluate_repr(pos_demos, neg_demos)

    print(f'{"Method":<20} {"P_sound":>8} {"R_sound":>8} {"F1_sound":>9}  '
          f'{"P_comp":>8} {"R_comp":>8} {"F1_comp":>8}')
    print('-' * 75)
    print(f'{"evaluate()":20} {p_s:>8.4f} {r_s:>8.4f} {f1_s:>9.4f}  '
          f'{p_c:>8.4f} {r_c:>8.4f} {f1_c:>8.4f}')
    print(f'{"evaluate_repr()":20} {p_s2:>8.4f} {r_s2:>8.4f} {f1_s2:>9.4f}  '
          f'{p_c2:>8.4f} {r_c2:>8.4f} {f1_c2:>8.4f}')

    # Highlight differences
    print()
    diff_p_c  = abs(p_c  - p_c2)
    diff_r_c  = abs(r_c  - r_c2)
    diff_f1_c = abs(f1_c - f1_c2)
    if diff_p_c < 1e-6 and diff_r_c < 1e-6:
        print('  \u2713 Both methods agree on complete model metrics.')
    else:
        print(f'  \u2192 Complete model differs — \u0394P_c={diff_p_c:.4f}  \u0394R_c={diff_r_c:.4f}  \u0394F1_c={diff_f1_c:.4f}')
        if diff_r_c > 1e-6:
            if r_c < r_c2:
                print('    evaluate_repr() has HIGHER complete recall — representative covers more positives')
            else:
                print('    evaluate_repr() has LOWER complete recall — representative misses some positives')
                print('    (expected before convergence: algorithm guarantees existential, not specific hyp)')

    # Convergence status
    print()
    print('  Per-action status (evaluate_repr):')
    for a, s in status.items():
        tag = '\u2705 converged' if s['converged'] else f'\u23f3 {s["n_hyps"]} hyps remaining'
        print(f'    {a}: {tag}')

    print()
    c1_ok  = abs(p_s  - 1.0) < 1e-9
    c2_ok  = abs(r_c  - 1.0) < 1e-9
    c1_ok2 = abs(p_s2 - 1.0) < 1e-9
    c2_ok2 = abs(r_c2 - 1.0) < 1e-9
    print(f'  C1 (P_sound=1): evaluate()={chr(10003) if c1_ok else chr(10007)}  evaluate_repr()={chr(10003) if c1_ok2 else chr(10007)}')
    print(f'  C2 (R_comp=1) : evaluate()={chr(10003) if c2_ok else chr(10007)}  evaluate_repr()={chr(10003) if c2_ok2 else chr(10007)}')


print('_compare_evaluations() defined.')


_compare_evaluations() defined.


In [14]:
_compare_evaluations(
    all_fluents, all_actions, static_fluents,
    pos_demos, neg_demos,
    label='Mockup domain — evaluate() vs evaluate_repr()'
)



  Mockup domain — evaluate() vs evaluate_repr()
Method                P_sound  R_sound  F1_sound    P_comp   R_comp  F1_comp
---------------------------------------------------------------------------
evaluate()             1.0000   1.0000    1.0000    1.0000   1.0000   1.0000
evaluate_repr()        1.0000   1.0000    1.0000    1.0000   1.0000   1.0000

  ✓ Both methods agree on complete model metrics.

  Per-action status (evaluate_repr):
    pickup: ⏳ 1 hyps remaining

  C1 (P_sound=1): evaluate()=✓  evaluate_repr()=✓
  C2 (R_comp=1) : evaluate()=✓  evaluate_repr()=✓


In [15]:
_compare_evaluations(
    b_fluents, b_actions, b_static,
    all_pos_blocks, all_neg_blocks,
    label='Blocks domain (extended, 20 problems) — evaluate() vs evaluate_repr()'
)



  Blocks domain (extended, 20 problems) — evaluate() vs evaluate_repr()
Method                P_sound  R_sound  F1_sound    P_comp   R_comp  F1_comp
---------------------------------------------------------------------------
evaluate()             1.0000   1.0000    1.0000    1.0000   1.0000   1.0000
evaluate_repr()        1.0000   1.0000    1.0000    1.0000   1.0000   1.0000

  ✓ Both methods agree on complete model metrics.

  Per-action status (evaluate_repr):
    pickup: ⏳ 1 hyps remaining
    putdown: ⏳ 1 hyps remaining
    stack: ⏳ 1 hyps remaining
    unstack: ⏳ 1 hyps remaining

  C1 (P_sound=1): evaluate()=✓  evaluate_repr()=✓
  C2 (R_comp=1) : evaluate()=✓  evaluate_repr()=✓
